# Ultra Marathon Running — Exploratory Data Analysis

Exploratory data analysis of **100 km ultra-marathon races held in Poland between 2010 and 2018**.

The notebook:
- downloads and loads the source dataset,
- filters the global dataset to the project scope,
- cleans and standardizes selected fields,
- calculates athlete age at the time of each race,
- validates dates and speed values,
- explores participation and performance patterns.

## 1. Imports

In [ ]:
from pathlib import Path
import zipfile

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## 2. Data Acquisition

In [ ]:
# Requires Kaggle API credentials configured on the local machine.
# Download can be skipped if the CSV already exists.

dataset_slug = "aiaiaidavid/the-big-dataset-of-ultra-marathon-running"
zip_path = Path("the-big-dataset-of-ultra-marathon-running.zip")
csv_path = Path("TWO_CENTURIES_OF_UM_RACES.csv")

if not csv_path.exists():
    if not zip_path.exists():
        !kaggle datasets download {dataset_slug}

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall()

print(f"Dataset file: {csv_path.resolve()}")

In [ ]:
df = pd.read_csv(csv_path, low_memory=False)

print(f"Raw dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
df.dtypes

## 3. Filtering & Project Scope

In [ ]:
# Keep only Polish 100 km events from 2010 through 2018.
# .copy() avoids chained-assignment / SettingWithCopyWarning issues later.

scope_mask = (
    df["Event distance/length"].eq("100km")
    & df["Year of event"].between(2010, 2018, inclusive="both")
    & df["Event name"].str.contains(r"\(POL\)", regex=True, na=False)
)

df2 = df.loc[scope_mask].copy()

print(f"Rows after scope filtering: {len(df2):,}")
print(f"Years: {df2['Year of event'].min()}–{df2['Year of event'].max()}")
print(f"Unique events: {df2['Event name'].nunique():,}")

## 4. Data Cleaning & Feature Engineering

In [ ]:
# Remove the redundant country suffix from event names.
df2["Event name"] = (
    df2["Event name"]
    .str.replace(r"\(POL\)", "", regex=True)
    .str.strip()
)

In [ ]:
# Athlete age should represent age at the time of the event,
# not age relative to the current year.
#
# Birth year is available, but exact birth date is not, so this is
# an approximate age in completed calendar years.

df2["athlete_age"] = (
    df2["Year of event"] - df2["Athlete year of birth"]
)

print(f"Missing athlete ages: {df2['athlete_age'].isna().sum():,}")

In [ ]:
# Remove the trailing "h" marker from performance strings while
# preserving the HH:MM:SS value.
df2["Athlete performance"] = (
    df2["Athlete performance"]
    .astype("string")
    .str.replace(r"\s*h\s*$", "", regex=True)
    .str.strip()
)

# Convert average speed to numeric.
# Invalid values become NaN and are handled explicitly below.
df2["Athlete average speed"] = pd.to_numeric(
    df2["Athlete average speed"],
    errors="coerce",
)

print(
    "Invalid / missing average-speed values:",
    f"{df2['Athlete average speed'].isna().sum():,}",
)

In [ ]:
# Event dates may be either a single date:
#   12.05.2018
# or a date range:
#   16.-17.03.2018
#
# Extract the final full DD.MM.YYYY date from either format.
# Unlike the previous split('-') approach, this does not discard
# single-date events.

date_text = df2["Event dates"].astype("string")

extracted_date = date_text.str.extract(
    r"(\d{1,2}\.\d{1,2}\.\d{4})\s*$",
    expand=False,
)

df2["race_day"] = pd.to_datetime(
    extracted_date,
    format="%d.%m.%Y",
    errors="coerce",
)

print(f"Unparsed event dates: {df2['race_day'].isna().sum():,}")

In [ ]:
# Data-quality summary before removing unusable rows.
quality_summary = pd.Series({
    "rows_before_cleaning": len(df2),
    "missing_age": df2["athlete_age"].isna().sum(),
    "missing_or_invalid_speed": df2["Athlete average speed"].isna().sum(),
    "unparsed_date": df2["race_day"].isna().sum(),
    "speed_above_50_kmh": (df2["Athlete average speed"] > 50).sum(),
})

quality_summary

In [ ]:
# Remove rows that cannot support the planned age / speed / date analysis.
df2 = df2.dropna(
    subset=["athlete_age", "Athlete average speed", "race_day"]
).copy()

# Conservative data-quality threshold:
# speeds above 50 km/h are implausible for a 100 km running event
# and are treated as corrupted records.
df2 = df2.loc[df2["Athlete average speed"] <= 50].copy()

df2["athlete_age"] = df2["athlete_age"].astype(int)

print(f"Rows after cleaning: {len(df2):,}")

In [ ]:
# Remove columns no longer needed after feature engineering.
df2 = df2.drop(
    columns=[
        "Athlete club",
        "Athlete country",
        "Athlete year of birth",
        "Athlete age category",
        "Event dates",
    ]
)

df2 = df2.rename(columns={
    "Year of event": "event_year",
    "Event name": "race_name",
    "Event distance/length": "race_length",
    "Event number of finishers": "race_number_of_finishers",
    "Athlete performance": "athlete_performance",
    "Athlete gender": "athlete_gender",
    "Athlete average speed": "athlete_average_speed",
    "Athlete ID": "athlete_id",
})

df2 = df2[
    [
        "event_year",
        "race_day",
        "race_name",
        "race_length",
        "race_number_of_finishers",
        "athlete_performance",
        "athlete_gender",
        "athlete_average_speed",
        "athlete_id",
        "athlete_age",
    ]
].reset_index(drop=True)

df2.head()

In [ ]:
print(f"Final dataset shape: {df2.shape[0]:,} rows × {df2.shape[1]} columns")
print(f"Year range: {df2['event_year'].min()}–{df2['event_year'].max()}")
print(f"Age range: {df2['athlete_age'].min()}–{df2['athlete_age'].max()}")
print(
    "Average-speed range:",
    f"{df2['athlete_average_speed'].min():.2f}–"
    f"{df2['athlete_average_speed'].max():.2f} km/h",
)

df2.isna().sum()

## 5. Exploratory Analysis

### Athlete Average Speed Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    data=df2,
    x="athlete_average_speed",
    bins=35,
    kde=True,
)
plt.title("Athlete Average Speed — Polish 100 km Events")
plt.xlabel("Average speed (km/h)")
plt.ylabel("Number of race results")
plt.show()

### Participation by Gender

In [ ]:
gender_counts = (
    df2["athlete_gender"]
    .value_counts(dropna=False)
    .rename_axis("athlete_gender")
    .reset_index(name="race_results")
)

gender_counts

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(
    data=df2,
    x="athlete_gender",
)
plt.title("Race Results by Gender")
plt.xlabel("Gender")
plt.ylabel("Number of race results")
plt.show()

### Average Speed by Gender

In [ ]:
df2.groupby("athlete_gender")["athlete_average_speed"].agg(
    ["mean", "median", "count"]
).sort_values("mean", ascending=False)

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df2,
    x="athlete_gender",
    y="athlete_average_speed",
)
plt.title("Average Speed Distribution by Gender")
plt.xlabel("Gender")
plt.ylabel("Average speed (km/h)")
plt.show()

### Age vs Performance

In [ ]:
# Restrict the comparison to ages with at least 20 observations
# to reduce noise from very small groups.

age_speed = (
    df2.groupby("athlete_age")["athlete_average_speed"]
    .agg(mean_speed="mean", median_speed="median", observations="count")
    .query("observations >= 20")
    .sort_index()
)

age_speed.head(15)

In [ ]:
plt.figure(figsize=(11, 5))
sns.lineplot(
    data=age_speed.reset_index(),
    x="athlete_age",
    y="mean_speed",
)
plt.title("Mean Average Speed by Athlete Age")
plt.xlabel("Athlete age at event")
plt.ylabel("Mean average speed (km/h)")
plt.show()

### Participation Over Time

In [ ]:
yearly_results = (
    df2.groupby("event_year")
    .size()
    .rename("race_results")
    .reset_index()
)

yearly_results

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=yearly_results,
    x="event_year",
    y="race_results",
    marker="o",
)
plt.title("100 km Race Results by Year")
plt.xlabel("Event year")
plt.ylabel("Number of race results")
plt.show()

### Most Represented Events

In [ ]:
event_summary = (
    df2.groupby("race_name")
    .agg(
        race_results=("athlete_id", "size"),
        editions=("event_year", "nunique"),
        mean_speed=("athlete_average_speed", "mean"),
    )
    .sort_values("race_results", ascending=False)
)

event_summary.head(10)

## 6. Notes

- `athlete_age` is calculated as `event_year - birth_year`, because the dataset does not contain exact dates of birth.
- `race_day` represents the final date in an event date field. For a multi-day range such as `16.-17.03.2018`, the stored date is `17.03.2018`.
- The 50 km/h speed threshold is used only to remove clearly corrupted values; it is not a performance classification.
- All final row counts should be taken from a fresh top-to-bottom notebook run after cleaning.